# Approach 2: Anchor-Based AKI Risk Stratification

Aligned with **Liu et al. 2018 (AMIA Proceedings, PMC5977670)**.

**Anchor points:**
- AKI patients: first SCr meeting KDIGO criteria
- Non-AKI patients: discharge − 1 day (fixed)

**Feature collection:** admission → anchor − LOOKBACK_HOURS (AKI) / anchor (non-AKI)

**Exclusions (Liu et al. aligned):**
- Age outside 18–64 years
- SCr > 1.3 mg/dL within 24h of admission
- Feature window before admission

**Features:** demographics, labs, vitals, comorbidities, medications (nephrotoxic exposure)

**Lead time:** `LOOKBACK_HOURS=48` primary (ADQI); `LOOKBACK_HOURS=24` secondary


## 1. Setup & Authentication

In [ ]:
!pip install google-cloud-bigquery pandas numpy matplotlib seaborn db-dtypes --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
from google.colab import auth
import os, warnings
warnings.filterwarnings('ignore')
auth.authenticate_user()
print('✓ Authentication successful!')


✓ Authentication successful!


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
PROJECT_ID              = 'mimic-aki-483914'           # ← UPDATE IF NEEDED
DATASET                 = 'physionet-data.mimiciv_3_1'

LOOKBACK_HOURS          = 24     # lead time: 48h primary, 24h secondary

# Age restriction:
# Liu et al. 2018 used 18-64. Relaxed to 18+ here (aligned to Phase 2/3):
#   (1) MIMIC-IV skews older than Liu et al. dataset
#   (2) AKI incidence rises with age — capping at 64 halves expected prevalence
#   (3) site_A (ICU archetype) targets 35% AKI prevalence -- needs a large
#       AKI-positive pool to sample from; age cap shrinks that pool
AGE_MIN                 = 18
AGE_MAX                 = None   # None = no upper age limit
SCR_ADMISSION_THRESHOLD = 1.3    # mg/dL — kept for reference; exclusion disabled below (aligned to Phase 2/3)

OUTPUT_CSV = f'aki_anchor_based_{LOOKBACK_HOURS}h_lookback.csv'

client = bigquery.Client(project=PROJECT_ID)

print('='*70)
print('APPROACH 2: ANCHOR-BASED AKI RISK STRATIFICATION')
print('Aligned with Liu et al. 2018 (AMIA, PMC5977670); exclusions relaxed to match Phase 2/3')
print('='*70)
print(f'  Lookback:        {LOOKBACK_HOURS}h')
age_str = f'{AGE_MIN}+' if AGE_MAX is None else f'{AGE_MIN}-{AGE_MAX}'
print(f'  Age range:       {age_str}')
print(f'  SCr threshold:   >{SCR_ADMISSION_THRESHOLD} mg/dL at admission — NOT excluded (see Step 5A)')
print(f'  Output:          {OUTPUT_CSV}')
print('='*70)


APPROACH 2: ANCHOR-BASED AKI RISK STRATIFICATION
Aligned with Liu et al. 2018 (AMIA, PMC5977670); exclusions relaxed to match Phase 2/3
  Lookback:        24h
  Age range:       18+
  SCr threshold:   >1.3 mg/dL at admission — NOT excluded (see Step 5A)
  Output:          aki_anchor_based_24h_lookback.csv


## 2. Last Hospital Admission per Patient
*(Reused from Approach 1)*

In [ ]:
%%time
print('STEP 1: LAST HOSPITAL ADMISSION PER PATIENT')

query_last_admission = f"""
WITH ranked AS (
  SELECT subject_id, hadm_id, admittime, dischtime, admission_type, insurance,
         ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY admittime DESC) AS rn
  FROM `{DATASET}_hosp.admissions`
)
SELECT subject_id, hadm_id, admittime, dischtime, admission_type, insurance
FROM ranked WHERE rn = 1
ORDER BY subject_id
"""
df_last_encounters = client.query(query_last_admission).to_dataframe()
df_last_encounters['stay_hours'] = (
    (df_last_encounters['dischtime'] - df_last_encounters['admittime'])
    .dt.total_seconds() / 3600
)
# Minimum stay: at least LOOKBACK_HOURS so a feature window exists
before = len(df_last_encounters)
df_last_encounters = df_last_encounters[
    df_last_encounters['stay_hours'] >= LOOKBACK_HOURS
].copy()
print(f'  ✓ {before:,} total  →  {len(df_last_encounters):,} after >{LOOKBACK_HOURS}h stay filter')
df_last_encounters.head()


STEP 1: LAST HOSPITAL ADMISSION PER PATIENT
  ✓ 223,452 total  →  171,462 after >24h stay filter
CPU times: user 4.33 s, sys: 138 ms, total: 4.46 s
Wall time: 18.7 s


,subject_id,hadm_id,admittime,dischtime,admission_type,insurance,stay_hours
0,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,EW EMER.,Medicaid,42.100000
4,10000117,27988844,2183-09-18 18:10:00,2183-09-21 16:30:00,OBSERVATION ADMIT,Medicaid,70.333333
8,10000560,28979390,2189-10-15 10:30:00,2189-10-17 15:00:00,SURGICAL SAME DAY ADMISSION,Private,52.500000
10,10000690,26146595,2152-01-28 23:40:00,2152-01-30 15:56:00,EW EMER.,Medicare,40.266667
11,10000719,24558333,2140-04-15 00:14:00,2140-04-18 12:29:00,URGENT,Private,84.250000


## 3. Demographics
*(Reused from Approach 1)*

In [ ]:
%%time
print('STEP 2: DEMOGRAPHICS')

df_patients = client.query(
    f"SELECT subject_id, gender, anchor_age, dod FROM `{DATASET}_hosp.patients`"
).to_dataframe()
df_demographics = df_last_encounters.merge(df_patients, on='subject_id', how='left')
df_demographics['age_at_admission'] = df_demographics['anchor_age']
print(f'  ✓ {len(df_demographics):,} patients  |  mean age {df_demographics["age_at_admission"].mean():.1f}')
df_demographics.head()

STEP 2: DEMOGRAPHICS
  ✓ 171,462 patients  |  mean age 57.6
CPU times: user 1.98 s, sys: 106 ms, total: 2.09 s
Wall time: 12.5 s


,subject_id,hadm_id,admittime,dischtime,admission_type,insurance,stay_hours,gender,anchor_age,dod,age_at_admission
0,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,EW EMER.,Medicaid,42.100000,F,52,2180-09-09,52
1,10000117,27988844,2183-09-18 18:10:00,2183-09-21 16:30:00,OBSERVATION ADMIT,Medicaid,70.333333,F,48,NaT,48
2,10000560,28979390,2189-10-15 10:30:00,2189-10-17 15:00:00,SURGICAL SAME DAY ADMISSION,Private,52.500000,F,53,NaT,53
3,10000690,26146595,2152-01-28 23:40:00,2152-01-30 15:56:00,EW EMER.,Medicare,40.266667,F,86,2152-01-30,86
4,10000719,24558333,2140-04-15 00:14:00,2140-04-18 12:29:00,URGENT,Private,84.250000,F,34,NaT,34


## 3A. Age Restriction

Liu et al. 2018 restrict to **age 18–64** at admission.


In [ ]:
print('STEP 3A: AGE RESTRICTION')

before = len(df_demographics)
if AGE_MIN is not None:
    df_demographics = df_demographics[
        df_demographics['age_at_admission'] >= AGE_MIN
    ].copy()
if AGE_MAX is not None:
    df_demographics = df_demographics[
        df_demographics['age_at_admission'] <= AGE_MAX
    ].copy()
print(f'  Before: {before:,}  After: {len(df_demographics):,}  '
      f'Removed: {before - len(df_demographics):,}')
age_str = f'{AGE_MIN}+' if AGE_MAX is None else f'{AGE_MIN}-{AGE_MAX}'
print(f'  Age filter applied: {age_str}')
print(f'  Mean age: {df_demographics["age_at_admission"].mean():.1f} years')
print(f'  Age range: {df_demographics["age_at_admission"].min():.0f} - '
      f'{df_demographics["age_at_admission"].max():.0f}')


STEP 3A: AGE RESTRICTION
  Before: 171,462  After: 171,462  Removed: 0
  Age filter applied: 18+
  Mean age: 57.6 years
  Age range: 18 - 91


## 4. Comorbidities
*(Reused from Approach 1)*

In [ ]:
%%time
print('STEP 3: COMORBIDITIES')

hadm_ids = df_demographics['hadm_id'].tolist()
query_comorbidities = f"""
SELECT hadm_id,
    MAX(CASE WHEN icd_code LIKE 'E11%' OR icd_code LIKE '250%' THEN 1 ELSE 0 END) AS has_diabetes,
    MAX(CASE WHEN icd_code LIKE 'I10%' OR icd_code LIKE '401%' THEN 1 ELSE 0 END) AS has_hypertension,
    MAX(CASE WHEN icd_code LIKE 'I50%' OR icd_code LIKE '428%' THEN 1 ELSE 0 END) AS has_chf,
    MAX(CASE WHEN icd_code LIKE 'A41%' OR icd_code LIKE '038%' THEN 1 ELSE 0 END) AS has_sepsis,
    MAX(CASE WHEN icd_code LIKE 'K70%' OR icd_code LIKE 'K74%' OR icd_code LIKE '571%' THEN 1 ELSE 0 END) AS has_liver_disease,
    MAX(CASE WHEN icd_code LIKE 'C%'   OR (icd_code >= '140' AND icd_code < '210')  THEN 1 ELSE 0 END) AS has_cancer,
    MAX(CASE WHEN icd_code LIKE 'N18%' OR icd_code LIKE '585%' THEN 1 ELSE 0 END) AS has_ckd
FROM `{DATASET}_hosp.diagnoses_icd`
WHERE hadm_id IN UNNEST(@hadm_ids)
GROUP BY hadm_id
"""
job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter('hadm_ids', 'INT64', hadm_ids)]
)
df_comorbidities = client.query(query_comorbidities, job_config=job_config).to_dataframe()
df_demographics  = df_demographics.merge(df_comorbidities, on='hadm_id', how='left')
for col in ['has_diabetes','has_hypertension','has_chf','has_sepsis',
            'has_liver_disease','has_cancer','has_ckd']:
    df_demographics[col] = df_demographics[col].fillna(0).astype(int)
print(f'  ✓ Comorbidities merged for {len(df_demographics):,} patients')

STEP 3: COMORBIDITIES
  ✓ Comorbidities merged for 171,462 patients
CPU times: user 2.16 s, sys: 94.3 ms, total: 2.25 s
Wall time: 18.2 s


## 5. All Creatinine Measurements
*(Reused from Approach 1)*

In [ ]:
%%time
print('STEP 4: ALL CREATININE MEASUREMENTS')

CACHE_SCR = 'cached_creatinine_approach2.csv'
subject_ids      = df_demographics['subject_id'].tolist()
earliest_admit   = df_demographics['admittime'].min()
latest_discharge = df_demographics['dischtime'].max()
lookback_date    = earliest_admit - pd.Timedelta(days=365)

if os.path.exists(CACHE_SCR):
    print('  ✓ Loading from cache...')
    df_scr_all = pd.read_csv(CACHE_SCR, parse_dates=['charttime'])
else:
    query_scr = f"""
    SELECT le.subject_id, le.hadm_id, le.charttime, le.valuenum AS creatinine_mg_dl
    FROM `{DATASET}_hosp.labevents` le
    WHERE le.subject_id IN UNNEST(@subject_ids)
        AND le.itemid = 50912
        AND le.valuenum IS NOT NULL AND le.valuenum > 0 AND le.valuenum < 20
        AND DATE(le.charttime) >= DATE(@lookback_date)
        AND DATE(le.charttime) <= DATE(@latest_discharge)
        AND le.charttime >= @lookback_date AND le.charttime <= @latest_discharge
    ORDER BY le.subject_id, le.charttime
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ArrayQueryParameter('subject_ids',    'INT64',    subject_ids),
        bigquery.ScalarQueryParameter('lookback_date', 'DATETIME', lookback_date),
        bigquery.ScalarQueryParameter('latest_discharge','DATETIME', latest_discharge),
    ])
    df_scr_all = client.query(query_scr, job_config=job_config).to_dataframe()
    df_scr_all.to_csv(CACHE_SCR, index=False)

print(f'  ✓ {len(df_scr_all):,} creatinine measurements  |  {df_scr_all["subject_id"].nunique():,} patients')

STEP 4: ALL CREATININE MEASUREMENTS
  ✓ 3,659,210 creatinine measurements  |  168,255 patients
CPU times: user 46.5 s, sys: 809 ms, total: 47.4 s
Wall time: 2min 44s


## 5A. SCr Admission Exclusion

Liu et al. 2018 exclude patients with SCr > 1.3 mg/dL within 24h of admission
(pre-existing kidney dysfunction — hospital-acquired AKI cannot be reliably distinguished).


In [ ]:
%%time
# ── STEP 5A: SCr ADMISSION EXCLUSION ── DISABLED (aligned to Phase 2/3) ──
# Liu et al. 2018 exclude SCr > 1.3 mg/dL within 24h of admission
# COMMENTED OUT: retaining these patients to increase AKI prevalence
# and better represent real-world ward population including mild CKD --
# matches Phase 2/3's documented reasoning.

# hadm_ids_scr = df_demographics['hadm_id'].tolist()
# subject_ids_scr = df_demographics['subject_id'].tolist()
#
# query_admission_scr = f"""
# SELECT le.subject_id, le.hadm_id, le.valuenum AS scr_admission
# FROM `{DATASET}_hosp.labevents` le
# JOIN (
#     SELECT subject_id, hadm_id, admittime
#     FROM `{DATASET}_hosp.admissions`
#     WHERE hadm_id IN UNNEST(@hadm_ids)
# ) adm ON le.subject_id = adm.subject_id
# WHERE le.subject_id IN UNNEST(@subject_ids)
#     AND le.itemid = 50912
#     AND le.valuenum IS NOT NULL AND le.valuenum > 0
#     AND le.charttime >= adm.admittime
#     AND le.charttime <= TIMESTAMP_ADD(adm.admittime, INTERVAL 24 HOUR)
# QUALIFY ROW_NUMBER() OVER (
#     PARTITION BY le.subject_id, le.hadm_id ORDER BY le.charttime ASC
# ) = 1
# """
# job_config = bigquery.QueryJobConfig(query_parameters=[
#     bigquery.ArrayQueryParameter('hadm_ids',    'INT64', hadm_ids_scr),
#     bigquery.ArrayQueryParameter('subject_ids', 'INT64', subject_ids_scr),
# ])
# df_admission_scr = client.query(query_admission_scr, job_config=job_config).to_dataframe()
#
# abnormal_scr = set(
#     df_admission_scr[
#         df_admission_scr['scr_admission'] > SCR_ADMISSION_THRESHOLD
#     ]['hadm_id']
# )
# before = len(df_demographics)
# df_demographics = df_demographics[
#     ~df_demographics['hadm_id'].isin(abnormal_scr)
# ].copy()
# print(f'  Excluded (SCr > {SCR_ADMISSION_THRESHOLD} mg/dL at admission): '
#       f'{before - len(df_demographics):,}')

print(f'  SCr admission exclusion SKIPPED — retaining all {len(df_demographics):,} patients')


  SCr admission exclusion SKIPPED — retaining all 171,462 patients
CPU times: user 35 µs, sys: 1 µs, total: 36 µs
Wall time: 39.3 µs


## 6. Identify CKD Patients
*(Reused from Approach 1)*

In [ ]:
# CKD patients identified here, then conditionally excluded in the next
# cell (Step 5): per the KDIGO baseline-SCr flowchart, only CKD patients
# who ALSO lack any SCr measurement in the year before admission are
# dropped -- CKD patients with recent SCr still get a real, measured
# baseline like everyone else.
ckd_patients_set = set(df_demographics[df_demographics['has_ckd'] == 1]['subject_id'])
print(f'  CKD patients (conditionally excluded if no SCr in past year): {len(ckd_patients_set):,}  '
      f'({len(ckd_patients_set)/len(df_demographics)*100:.1f}%)')


  CKD patients (informational only, not excluded): 22,735  (13.3%)


## 7. Calculate Baseline SCr
*(Reused from Approach 1)*

In [ ]:
%%time
print('STEP 5: BASELINE SCr')

def calculate_mdrd_baseline_scr(age, sex, race='WHITE', target_egfr=75):
    sex_factor  = 0.742 if sex == 'F' else 1.0
    race_factor = 1.212 if race in ['BLACK', 'BLACK/AFRICAN AMERICAN'] else 1.0
    return (target_egfr / (175 * (age ** -0.203) * sex_factor * race_factor)) ** (-1/1.154)

def calculate_baseline_scr(row, df_scr_all, ckd_patients_set):
    # KDIGO baseline-SCr estimation, three-tier hierarchy:
    #   1. SCr within 7 days prior to admission (most recent) -> strong evidence
    #   2. SCr 7-365 days prior to admission (mean)            -> less strong evidence
    #   3. No SCr in the past year:
    #        - CKD history      -> drop (MDRD's eGFR=75 assumption is
    #          unreliable for known CKD; no dependable baseline available)
    #        - no CKD history   -> MDRD-estimated SCr (eGFR=75 mL/min/1.73m^2)
    # Every tier's estimate is additionally capped at the admission 24h SCr
    # value via min(), so a lower observed value never gets overridden by
    # an estimate.
    sid        = row['subject_id']
    admittime  = row['admittime']
    patient_scr = df_scr_all[df_scr_all['subject_id'] == sid].sort_values('charttime')

    scr_first_24h_min = patient_scr[
        (patient_scr['charttime'] >= admittime) &
        (patient_scr['charttime'] <= admittime + pd.Timedelta(hours=24))
    ]['creatinine_mg_dl'].min()
    scr_first_24h_min = None if pd.isna(scr_first_24h_min) else scr_first_24h_min

    scr_7d    = patient_scr[(patient_scr['charttime'] >= admittime - pd.Timedelta(days=7)) &
                             (patient_scr['charttime'] < admittime)]['creatinine_mg_dl']
    scr_365to7= patient_scr[(patient_scr['charttime'] >= admittime - pd.Timedelta(days=365)) &
                             (patient_scr['charttime'] < admittime - pd.Timedelta(days=7))]['creatinine_mg_dl']

    if len(scr_7d) > 0:
        ref = scr_7d.iloc[-1]
        baseline = min(ref, scr_first_24h_min) if scr_first_24h_min else ref
        method   = 'most_recent_7d'
    elif len(scr_365to7) > 0:
        ref = scr_365to7.mean()
        baseline = min(ref, scr_first_24h_min) if scr_first_24h_min else ref
        method   = 'avg_365to7'
    elif sid in ckd_patients_set:
        baseline = None
        method   = 'dropped_ckd_no_recent_scr'
    else:
        mdrd = calculate_mdrd_baseline_scr(row['age_at_admission'], row['gender'])
        baseline = min(mdrd, scr_first_24h_min) if scr_first_24h_min else mdrd
        method   = 'mdrd'

    return pd.Series({'baseline_scr': baseline, 'baseline_method': method})

baseline_results = df_demographics.apply(
    lambda r: calculate_baseline_scr(r, df_scr_all, ckd_patients_set), axis=1
)
df_demographics = pd.concat([df_demographics, baseline_results], axis=1)

n_before      = len(df_demographics)
n_dropped_ckd = (df_demographics['baseline_method'] == 'dropped_ckd_no_recent_scr').sum()
df_demographics = df_demographics[df_demographics['baseline_scr'].notna()].reset_index(drop=True)
print(f'  Dropped (CKD history, no SCr in past year, per KDIGO baseline flow): {n_dropped_ckd:,}  '
      f'({n_dropped_ckd/n_before*100:.1f}%)')
print(f'  Baseline calculated for {len(df_demographics):,} patients')
print(df_demographics['baseline_method'].value_counts())


STEP 5: BASELINE SCr
  ✓ Baseline calculated for 171,462 patients (no CKD exclusion)
baseline_method
most_recent_7d    121753
mdrd               25349
avg_365to7         24360
Name: count, dtype: int64
CPU times: user 13min 14s, sys: 835 ms, total: 13min 15s
Wall time: 13min 15s


## 8. Extract All Labs During Admission

In [ ]:
%%time
print('STEP 6: ALL LABS DURING ADMISSION')

LAB_ITEMIDS = {
    'creatinine':  [50912],
    'bun':         [51006],
    'lactate':     [50813],
    'sodium':      [50983],
    'potassium':   [50971],
    'bicarbonate': [50882],
    'wbc':         [51301],
    'hemoglobin':  [51222, 50811],
    'platelets':   [51265, 51704],
    'glucose':     [50931, 50809],
    'albumin':     [50862],
    'bilirubin':   [50885],
}
all_lab_itemids = [iid for ids in LAB_ITEMIDS.values() for iid in ids]
itemid_to_lab   = {iid: name for name, ids in LAB_ITEMIDS.items() for iid in ids}

CACHE_LABS = 'cached_labs_approach2.csv'
hadm_ids   = df_demographics['hadm_id'].tolist()

if os.path.exists(CACHE_LABS):
    print('  ✓ Loading labs from cache...')
    df_labs_all = pd.read_csv(CACHE_LABS, parse_dates=['charttime'])
else:
    query_labs = f"""
    WITH adm AS (
        SELECT subject_id, hadm_id, admittime, dischtime
        FROM `{DATASET}_hosp.admissions`
        WHERE hadm_id IN UNNEST(@hadm_ids)
    )
    SELECT le.subject_id, le.hadm_id, le.itemid, le.charttime, le.valuenum
    FROM adm
    JOIN `{DATASET}_hosp.labevents` le
        ON adm.subject_id = le.subject_id
        AND le.charttime BETWEEN adm.admittime AND adm.dischtime
        AND le.itemid IN UNNEST(@itemids)
        AND le.valuenum IS NOT NULL AND le.valuenum > 0
        AND DATE(le.charttime) BETWEEN
            DATE_SUB(DATE(adm.admittime), INTERVAL 1 DAY)
            AND DATE_ADD(DATE(adm.dischtime), INTERVAL 1 DAY)
    ORDER BY le.subject_id, le.charttime
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ArrayQueryParameter('hadm_ids', 'INT64', hadm_ids),
        bigquery.ArrayQueryParameter('itemids',  'INT64', all_lab_itemids),
    ])
    df_labs_all = client.query(query_labs, job_config=job_config).to_dataframe()
    df_labs_all.to_csv(CACHE_LABS, index=False)

df_labs_all['feature_name'] = df_labs_all['itemid'].map(itemid_to_lab)
print(f'  ✓ {len(df_labs_all):,} lab measurements  |  {df_labs_all["subject_id"].nunique():,} patients')

STEP 6: ALL LABS DURING ADMISSION
  ✓ 9,850,694 lab measurements  |  154,126 patients
CPU times: user 2min 22s, sys: 556 ms, total: 2min 23s
Wall time: 7min 20s


## 9. Extract All Vitals During Admission

In [ ]:
%%time
print('STEP 7: ALL VITALS DURING ADMISSION')

VITAL_ITEMIDS = {
    'heart_rate':  [220045],
    'sbp':         [220179],
    'dbp':         [220180],
    'resp_rate':   [220210],
    'spo2':        [220277],
    'temperature': [223761, 223762],
    'gcs_total':   [220739],
}
all_vital_itemids = [iid for ids in VITAL_ITEMIDS.values() for iid in ids]
itemid_to_vital   = {iid: name for name, ids in VITAL_ITEMIDS.items() for iid in ids}

CACHE_VITALS = 'cached_vitals_approach2.csv'

if os.path.exists(CACHE_VITALS):
    print('  ✓ Loading vitals from cache...')
    df_vitals_all = pd.read_csv(CACHE_VITALS, parse_dates=['charttime'])
else:
    query_vitals = f"""
    WITH adm AS (
        SELECT subject_id, hadm_id, admittime, dischtime
        FROM `{DATASET}_hosp.admissions`
        WHERE hadm_id IN UNNEST(@hadm_ids)
    )
    SELECT ce.subject_id, ce.hadm_id, ce.itemid, ce.charttime, ce.valuenum
    FROM adm
    JOIN `{DATASET}_icu.chartevents` ce
        ON adm.subject_id = ce.subject_id
        AND ce.charttime BETWEEN adm.admittime AND adm.dischtime
        AND ce.itemid IN UNNEST(@itemids)
        AND ce.valuenum IS NOT NULL AND ce.valuenum > 0
        AND DATE(ce.charttime) BETWEEN
            DATE_SUB(DATE(adm.admittime), INTERVAL 1 DAY)
            AND DATE_ADD(DATE(adm.dischtime), INTERVAL 1 DAY)
    ORDER BY ce.subject_id, ce.charttime
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ArrayQueryParameter('hadm_ids', 'INT64', hadm_ids),
        bigquery.ArrayQueryParameter('itemids',  'INT64', all_vital_itemids),
    ])
    df_vitals_all = client.query(query_vitals, job_config=job_config).to_dataframe()
    df_vitals_all.to_csv(CACHE_VITALS, index=False)

df_vitals_all['feature_name'] = df_vitals_all['itemid'].map(itemid_to_vital)
print(f'  ✓ {len(df_vitals_all):,} vital measurements  |  {df_vitals_all["subject_id"].nunique():,} patients')

STEP 7: ALL VITALS DURING ADMISSION
  ✓ 22,863,323 vital measurements  |  43,740 patients
CPU times: user 5min 29s, sys: 1.25 s, total: 5min 30s
Wall time: 16min 12s


## 10. Combine Labs and Vitals

In [ ]:
ALL_FEATURES = list(LAB_ITEMIDS.keys()) + list(VITAL_ITEMIDS.keys())

df_measurements = pd.concat([
    df_labs_all[['subject_id', 'hadm_id', 'charttime', 'feature_name', 'valuenum']],
    df_vitals_all[['subject_id', 'hadm_id', 'charttime', 'feature_name', 'valuenum']],
], ignore_index=True).sort_values(['subject_id', 'hadm_id', 'charttime'])

print(f'  ✓ Combined: {len(df_measurements):,} measurements  |  {len(ALL_FEATURES)} features')
print(f'  ✓ Features: {ALL_FEATURES}')

  ✓ Combined: 32,714,017 measurements  |  19 features
  ✓ Features: ['creatinine', 'bun', 'lactate', 'sodium', 'potassium', 'bicarbonate', 'wbc', 'hemoglobin', 'platelets', 'glucose', 'albumin', 'bilirubin', 'heart_rate', 'sbp', 'dbp', 'resp_rate', 'spo2', 'temperature', 'gcs_total']


## 11. Extract Medications During Admission

Liu et al. 2018: medications were the **strongest predictor** of AKI.
Features: `nephrotoxic_flag`, `nephrotoxic_count`, `n_distinct_meds`.
Note: aggregated up to `dischtime - 24h` here; filtered to `feature_cutoff` in Step 14.


In [ ]:
%%time
print('STEP 11: EXTRACT MEDICATIONS')

CACHE_MEDS = 'cached_meds_approach2.csv'
hadm_ids_med = df_demographics['hadm_id'].tolist()

NEPHROTOXIC_NAMES = [
    'ibuprofen','naproxen','ketorolac','indomethacin','diclofenac',
    'gentamicin','tobramycin','amikacin',
    'vancomycin',
    'iohexol','iopamidol','iodixanol',
    'lisinopril','enalapril','captopril','ramipril',
    'losartan','valsartan','irbesartan','olmesartan',
    'furosemide','bumetanide','torsemide',
    'tacrolimus','cyclosporine',
    'cisplatin','carboplatin',
]

if os.path.exists(CACHE_MEDS):
    print('  ✓ Loading from cache...')
    df_meds_all = pd.read_csv(CACHE_MEDS, parse_dates=['starttime'])
else:
    drug_conditions = ' OR '.join(
        [f"LOWER(pr.drug) LIKE '%{d}%'" for d in NEPHROTOXIC_NAMES]
    )
    query_meds = f"""
    WITH adm AS (
        SELECT subject_id, hadm_id, admittime, dischtime
        FROM `{DATASET}_hosp.admissions`
        WHERE hadm_id IN UNNEST(@hadm_ids)
    )
    SELECT pr.subject_id, pr.hadm_id, pr.starttime, pr.drug,
           CASE WHEN {drug_conditions} THEN 1 ELSE 0 END AS is_nephrotoxic
    FROM adm
    JOIN `{DATASET}_hosp.prescriptions` pr
        ON adm.subject_id = pr.subject_id
        AND pr.starttime BETWEEN adm.admittime AND adm.dischtime
    WHERE pr.drug IS NOT NULL
    ORDER BY pr.subject_id, pr.starttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter('hadm_ids','INT64',hadm_ids_med)]
    )
    df_meds_all = client.query(query_meds, job_config=job_config).to_dataframe()
    df_meds_all.to_csv(CACHE_MEDS, index=False)

print(f'  ✓ {len(df_meds_all):,} medication records  |  '
      f'{df_meds_all["subject_id"].nunique():,} patients')


STEP 11: EXTRACT MEDICATIONS
  ✓ 7,949,395 medication records  |  166,980 patients
CPU times: user 1min 54s, sys: 491 ms, total: 1min 54s
Wall time: 6min 10s


## 12. Detect AKI Onset and Assign Anchor Points

In [ ]:
%%time
print('STEP 8: DETECT AKI ONSET TIME AND ASSIGN ANCHOR POINTS')

# Filter SCr to admission period only
df_scr_adm = df_scr_all.merge(
    df_demographics[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'baseline_scr']],
    on=['subject_id', 'hadm_id'], how='inner'
)
df_scr_adm = df_scr_adm[
    (df_scr_adm['charttime'] >= df_scr_adm['admittime']) &
    (df_scr_adm['charttime'] <= df_scr_adm['dischtime'])
].copy().sort_values(['hadm_id', 'charttime'])

# ── KDIGO Criterion 2: SCr >= 1.5x baseline (vectorised) ─────────────────
df_scr_adm['aki_1_5x'] = (
    df_scr_adm['creatinine_mg_dl'] >= 1.5 * df_scr_adm['baseline_scr']
).astype(int)

# ── KDIGO Criterion 1: rise >= 0.3 mg/dL within any 48h window ───────────
def rolling_min_48h(group):
    vals  = group['creatinine_mg_dl'].values
    times = group['charttime'].values
    result = []
    for i in range(len(vals)):
        window_start = times[i] - np.timedelta64(48, 'h')
        prior = vals[(times >= window_start) & (times < times[i])]
        result.append(prior.min() if len(prior) > 0 else np.nan)
    return pd.Series(result, index=group.index)

df_scr_adm['prior_48h_min'] = (
    df_scr_adm.groupby('hadm_id', group_keys=False)
    .apply(rolling_min_48h)
)
df_scr_adm['aki_48h_rise'] = (
    (df_scr_adm['creatinine_mg_dl'] - df_scr_adm['prior_48h_min']) >= 0.3
).fillna(False).astype(int)

df_scr_adm['aki_flag'] = (
    (df_scr_adm['aki_1_5x'] == 1) | (df_scr_adm['aki_48h_rise'] == 1)
).astype(int)

# ── AKI patients: anchor = first KDIGO-positive SCr ──────────────────────
aki_onset = (
    df_scr_adm[df_scr_adm['aki_flag'] == 1]
    .groupby('hadm_id')['charttime']
    .min()
    .reset_index()
    .rename(columns={'charttime': 'aki_onset_time'})
)
df_demographics = df_demographics.merge(aki_onset, on='hadm_id', how='left')
df_demographics['AKI_label'] = df_demographics['aki_onset_time'].notna().astype(int)

# ── Non-AKI patients: anchor = last SCr measurement during admission ─────
# CHANGED (aligned to Phase 2/Phase 3 GPC-aligned methodology): previously
# used dischtime - 24h (discharge day - 1, fixed regardless of when SCr was
# actually last drawn). GPC anchors non-AKI patients to their last SCr
# measurement instead, so we align to that here: last_scr_time - 24h.
# Uses ALL SCr draws during admission (not just KDIGO-positive ones) --
# df_scr_adm is already filtered to [admittime, dischtime].
last_scr = (
    df_scr_adm
    .groupby('hadm_id')['charttime']
    .max()
    .reset_index()
    .rename(columns={'charttime': 'last_scr_time'})
)
df_demographics = df_demographics.merge(last_scr, on='hadm_id', how='left')

# ── Anchor points (Liu et al. 2018, PMC5977670; non-AKI anchor revised) ──
#
# AKI patients:     anchor = first KDIGO-positive SCr
# Non-AKI patients: anchor = last_scr_time - 24h  (last SCr - 1 day)
#
# Matches paper data collection window for AKI patients:
#   AKI:     [Admission_date, AKI_date - n]  where n = LOOKBACK_HOURS/24
# Non-AKI patients now anchor to their own last SCr draw, not a fixed
# discharge-relative cutoff -- matches Phase 2/Phase 3 GPC methodology.
#
# LOOKBACK_HOURS applies to AKI patients only as a lead-time buffer.
# Non-AKI patients have no AKI event so no lead-time gap is needed.
df_demographics['anchor_time'] = np.where(
    df_demographics['AKI_label'] == 1,
    df_demographics['aki_onset_time'],
    df_demographics['last_scr_time'] - pd.Timedelta(hours=24)
)
df_demographics['anchor_time'] = pd.to_datetime(df_demographics['anchor_time'])

# ── Feature cutoff ────────────────────────────────────────────────────────
# AKI:     feature_cutoff = anchor - LOOKBACK_HOURS  (lead-time buffer before onset)
# Non-AKI: feature_cutoff = anchor                   (last_scr_time - 24h IS the cutoff)
df_demographics['feature_cutoff'] = np.where(
    df_demographics['AKI_label'] == 1,
    pd.to_datetime(df_demographics['anchor_time']) - pd.Timedelta(hours=LOOKBACK_HOURS),
    pd.to_datetime(df_demographics['anchor_time'])
)
df_demographics['feature_cutoff'] = pd.to_datetime(df_demographics['feature_cutoff'])

n_aki    = df_demographics['AKI_label'].sum()
n_no_aki = (df_demographics['AKI_label'] == 0).sum()
print(f'  aki patients:           {n_aki:,}  ({n_aki/len(df_demographics)*100:.1f}%)')
print(f'  Non-AKI patients:       {n_no_aki:,}  ({n_no_aki/len(df_demographics)*100:.1f}%)')
print(f'  AKI anchor:             first KDIGO-positive SCr')
print(f'  AKI feature cutoff:     AKI onset - {LOOKBACK_HOURS}h')
print(f'  Non-AKI anchor:         last SCr during admission - 24h  (CHANGED, matches Phase 2/3)')
print(f'  Non-AKI feature cutoff: last SCr - 24h  (anchor = cutoff)')
print(f'  Missing anchor:         {pd.to_datetime(df_demographics["anchor_time"]).isna().sum():,}')
print(f'  Missing last_scr_time:  {df_demographics["last_scr_time"].isna().sum():,}  (non-AKI patients with no SCr during admission -- will need exclusion review)')


STEP 8: DETECT AKI ONSET TIME AND ASSIGN ANCHOR POINTS
  aki patients:           28,740  (16.8%)
  Non-AKI patients:       142,722  (83.2%)
  AKI anchor:             first KDIGO-positive SCr
  AKI feature cutoff:     AKI onset - 24h
  Non-AKI anchor:         last SCr during admission - 24h  (CHANGED, matches Phase 2/3)
  Non-AKI feature cutoff: last SCr - 24h  (anchor = cutoff)
  Missing anchor:         28,566
  Missing last_scr_time:  28,566  (non-AKI patients with no SCr during admission -- will need exclusion review)
CPU times: user 27.8 s, sys: 68.4 ms, total: 27.9 s
Wall time: 27.9 s


## 13. Exclusions

In [ ]:
%%time
print('STEP 9: EXCLUSIONS')

# feature_cutoff computed in Step 8 per group:
#   AKI:     anchor - LOOKBACK_HOURS
#   Non-AKI: anchor (= dischtime - 24h)
# No recomputation needed here.

df_demographics['hours_to_anchor'] = (
    (df_demographics['anchor_time'] - df_demographics['admittime'])
    .dt.total_seconds() / 3600
)

before = len(df_demographics)

# Exclude: no anchor (no SCr during admission)
df_demographics = df_demographics[df_demographics['anchor_time'].notna()].copy()
after_no_scr = len(df_demographics)

# Exclude: feature_cutoff falls before admission
# (AKI onset or discharge too close to admission for any features to exist)
df_demographics = df_demographics[
    df_demographics['feature_cutoff'] >= df_demographics['admittime']
].copy()
after_short = len(df_demographics)

print(f'  Starting patients:                    {before:,}')
print(f'  After removing no-SCr patients:       {after_no_scr:,}  '
      f'(removed {before - after_no_scr:,})')
print(f'  After removing invalid windows:       {after_short:,}  '
      f'(removed {after_no_scr - after_short:,})')
print(f'  Final cohort:                         {len(df_demographics):,}')
print(f'  AKI prevalence:                       '
      f'{df_demographics["AKI_label"].mean()*100:.1f}%')
print(f'  Mean hours to anchor (AKI):           '
      f'{df_demographics[df_demographics["AKI_label"]==1]["hours_to_anchor"].mean():.1f}h')
print(f'  Mean hours to anchor (non-AKI):       '
      f'{df_demographics[df_demographics["AKI_label"]==0]["hours_to_anchor"].mean():.1f}h')


STEP 9: EXCLUSIONS
  Starting patients:                    171,462
  After removing no-SCr patients:       142,896  (removed 28,566)
  After removing invalid windows:       115,404  (removed 27,492)
  Final cohort:                         115,404
  AKI prevalence:                       17.6%
  Mean hours to anchor (AKI):           111.4h
  Mean hours to anchor (non-AKI):       87.6h
CPU times: user 65.4 ms, sys: 2 µs, total: 65.4 ms
Wall time: 65.1 ms


## 14. Extract Features and Medications Up to Feature Cutoff

All measurements and medications filtered to `[admittime, feature_cutoff]` per patient.


In [ ]:
%%time
print('STEP 10: EXTRACT FEATURES UP TO FEATURE CUTOFF')

# Merge measurements with per-patient cutoff times
df_meas = df_measurements.merge(
    df_demographics[['hadm_id', 'admittime', 'feature_cutoff']],
    on='hadm_id', how='inner'
)

# Keep only measurements within [admittime, feature_cutoff]
df_meas = df_meas[
    (df_meas['charttime'] >= df_meas['admittime']) &
    (df_meas['charttime'] <= df_meas['feature_cutoff'])
].copy()

print(f'  ✓ Measurements within feature windows: {len(df_meas):,}')
print(f'  ✓ Patients with at least one measurement: '
      f'{df_meas["hadm_id"].nunique():,}')

# ── Compute summary stats per (hadm_id, feature_name) ─────────────────────
df_meas = df_meas.sort_values(['hadm_id', 'feature_name', 'charttime'])

agg_funcs = {
    'valuenum': ['last', 'min', 'max', 'mean'],
    'charttime': 'last'
}
df_feat = (
    df_meas.groupby(['hadm_id', 'feature_name'])
    .agg(agg_funcs)
    .reset_index()
)
df_feat.columns = ['hadm_id', 'feature_name',
                   'most_recent', 'min', 'max', 'mean', 'last_charttime']

# Merge feature_cutoff back to compute hours_since
df_feat = df_feat.merge(
    df_demographics[['hadm_id', 'feature_cutoff']], on='hadm_id', how='left'
)
df_feat['hours_since'] = (
    (df_feat['feature_cutoff'] - df_feat['last_charttime'])
    .dt.total_seconds() / 3600
).clip(lower=0)

# ── Pivot wide: one row per hadm_id ───────────────────────────────────────
df_wide = df_feat.pivot_table(
    index='hadm_id',
    columns='feature_name',
    values=['most_recent', 'min', 'max', 'mean', 'hours_since'],
    aggfunc='first'
)
df_wide.columns = [f'{feat}_{stat}' for stat, feat in df_wide.columns]
df_wide = df_wide.reset_index()

print(f'  ✓ Wide feature table: {df_wide.shape[0]:,} patients × {df_wide.shape[1]-1} features')


STEP 10: EXTRACT FEATURES UP TO FEATURE CUTOFF
  ✓ Measurements within feature windows: 14,652,882
  ✓ Patients with at least one measurement: 96,515
  ✓ Wide feature table: 96,515 patients × 95 features
CPU times: user 7.82 s, sys: 1.57 s, total: 9.38 s
Wall time: 9.41 s


In [ ]:
%%time
print('STEP 14B: AGGREGATE MEDICATIONS UP TO FEATURE CUTOFF')

# Now feature_cutoff exists — filter medications per patient
df_meds = df_meds_all.merge(
    df_demographics[['hadm_id', 'admittime', 'feature_cutoff']],
    on='hadm_id', how='inner'
)
df_meds = df_meds[
    (df_meds['starttime'] >= df_meds['admittime']) &
    (df_meds['starttime'] <= df_meds['feature_cutoff'])
].copy()

med_agg = df_meds.groupby('hadm_id').agg(
    nephrotoxic_count=('is_nephrotoxic', 'sum'),
    n_distinct_meds  =('drug', 'nunique'),
).reset_index()
med_agg['nephrotoxic_flag'] = (med_agg['nephrotoxic_count'] > 0).astype(int)

df_demographics = df_demographics.merge(med_agg, on='hadm_id', how='left')
df_demographics['nephrotoxic_count'] = df_demographics['nephrotoxic_count'].fillna(0).astype(int)
df_demographics['nephrotoxic_flag']  = df_demographics['nephrotoxic_flag'].fillna(0).astype(int)
df_demographics['n_distinct_meds']   = df_demographics['n_distinct_meds'].fillna(0).astype(int)

print(f'  ✓ nephrotoxic_flag=1: {df_demographics["nephrotoxic_flag"].sum():,} '
      f'({df_demographics["nephrotoxic_flag"].mean()*100:.1f}%)')
print(f'  ✓ Mean nephrotoxic_count: {df_demographics["nephrotoxic_count"].mean():.2f}')
print(f'  ✓ Mean n_distinct_meds:   {df_demographics["n_distinct_meds"].mean():.1f}')


STEP 14B: AGGREGATE MEDICATIONS UP TO FEATURE CUTOFF
  ✓ nephrotoxic_flag=1: 55,415 (48.0%)
  ✓ Mean nephrotoxic_count: 1.60
  ✓ Mean n_distinct_meds:   20.5
CPU times: user 1.27 s, sys: 251 ms, total: 1.52 s
Wall time: 1.52 s


## 15. Build Final Dataset

In [ ]:
%%time
print('STEP 15: BUILD FINAL DATASET')

# Static features from demographics
static_cols = [
    'hadm_id', 'subject_id', 'age_at_admission', 'gender',
    'admission_type', 'baseline_scr', 'baseline_method',
    'has_diabetes', 'has_hypertension', 'has_chf', 'has_sepsis',
    'has_liver_disease', 'has_cancer',
    'nephrotoxic_flag', 'nephrotoxic_count', 'n_distinct_meds',
    'hours_to_anchor', 'AKI_label',
    'admittime', 'anchor_time', 'feature_cutoff',
]
df_static = df_demographics[static_cols].copy()
df_static['gender'] = (df_static['gender'] == 'M').astype(int)
df_static['admission_type'] = pd.Categorical(df_static['admission_type']).codes

# Merge with dynamic features
df_final = df_static.merge(df_wide, on='hadm_id', how='left')
df_final['center_id'] = 0
df_final = df_final.reset_index(drop=True)

# Feature columns (exclude ids, meta, label)
meta_cols = ['hadm_id', 'subject_id', 'admittime', 'anchor_time',
             'feature_cutoff', 'baseline_method', 'AKI_label', 'center_id']
feature_cols = [c for c in df_final.columns if c not in meta_cols]

print(f'  ✓ Final cohort:      {len(df_final):,} patients')
print(f'  ✓ AKI prevalence:    {df_final["AKI_label"].mean()*100:.1f}%')
print(f'  ✓ Feature columns:   {len(feature_cols)}')
print(f'  ✓ Missing data rate: {df_final[feature_cols].isna().mean().mean()*100:.1f}% (avg across features)')


STEP 15: BUILD FINAL DATASET
  ✓ Final cohort:      115,404 patients
  ✓ AKI prevalence:    17.6%
  ✓ Feature columns:   109
  ✓ Missing data rate: 40.8% (avg across features)
CPU times: user 131 ms, sys: 16 ms, total: 147 ms
Wall time: 146 ms


## 15B. Leakage-Column Removal — Anchor-Selection Asymmetry

**Finding:** `hours_since` (per-lab recency) and `hours_to_anchor` (patient-level time-to-anchor) are dropped from the modeling feature set. Neither leaks *future* information at the individual-patient level -- both are computed strictly from data at/before `feature_cutoff`. The issue is **anchor-selection asymmetry** introduced in Step 8: AKI patients' anchor is a prospective, outcome-triggered event (first KDIGO-positive SCr); non-AKI patients' anchor is retrospective and whole-stay-dependent (last SCr draw across the entire admission, minus 24h). Because the two groups' anchors are selected by different logic, monitoring density near `feature_cutoff` differs systematically by construction -- not real clinical signal. Confirmed for the Phase 2/Phase 3 GPC-aligned cohort: recency alone (no lab values) gets 0.62 AUROC. Same mechanism applies here since this notebook now uses the same anchor construction.

(The downstream simulation script's own `FEATURE_GROUPS`-based column resolution independently excludes these same columns -- see `resolve_feature_columns()` in `mimic_ftl_simulation_phase1_archetype_post_leakage.py` -- so this notebook-level cleanup and that script-level filter are two independent safety nets, not one mechanism split across files.)

In [ ]:
print('STEP 15B: REMOVE LEAKAGE COLUMNS (hours_since, hours_to_anchor)')

# Identify leakage columns actually present in the modeling feature set
leak_cols = [c for c in feature_cols if c.endswith('_hours_since')]
if 'hours_to_anchor' in feature_cols:
    leak_cols.append('hours_to_anchor')

n_before = len(feature_cols)
feature_cols = [c for c in feature_cols if c not in leak_cols]
n_after = len(feature_cols)

print(f'  Removed {len(leak_cols)} leakage columns '
      f'({sum(c.endswith("_hours_since") for c in leak_cols)} *_hours_since '
      f'+ {"1" if "hours_to_anchor" in leak_cols else "0"} hours_to_anchor)')
print(f'  feature_cols: {n_before} -> {n_after}')

# NOTE: no FEATURE_GROUPS cleanup here -- that dict is defined in the
# separate downstream simulation script (mimic_ftl_simulation_phase1_
# archetype_post_leakage.py), not in this notebook's namespace. That
# script's resolve_feature_columns() already excludes *_hours_since and
# never lists hours_to_anchor, so it's independently protected -- this
# feature_cols cleanup and that script's protection are two separate,
# redundant safety nets, not one mechanism split across two places.


STEP 15B: REMOVE LEAKAGE COLUMNS (hours_since, hours_to_anchor)
  Removed 20 leakage columns (19 *_hours_since + 1 hours_to_anchor)
  feature_cols: 109 -> 89


## 16. Patient-Level Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

print('STEP 16: PATIENT-LEVEL TRAIN/TEST SPLIT (80/20 stratified)')

train_ids, test_ids = train_test_split(
    df_final['subject_id'],
    test_size=0.20,
    stratify=df_final['AKI_label'],
    random_state=42,
)
df_final['split'] = np.where(df_final['subject_id'].isin(train_ids), 'train', 'test')

train_df = df_final[df_final['split'] == 'train']
test_df  = df_final[df_final['split'] == 'test']

print(f'  ✓ Train: {len(train_df):,} patients  AKI={train_df["AKI_label"].mean()*100:.1f}%')
print(f'  ✓ Test:  {len(test_df):,} patients  AKI={test_df["AKI_label"].mean()*100:.1f}%')
print(f'  ✓ No patient overlap: {len(set(train_ids) & set(test_ids)) == 0}')


STEP 16: PATIENT-LEVEL TRAIN/TEST SPLIT (80/20 stratified)
  ✓ Train: 92,323 patients  AKI=17.6%
  ✓ Test:  23,081 patients  AKI=17.6%
  ✓ No patient overlap: True


## 17. Save

In [ ]:
print('STEP 17: SAVE')

save_cols = ['subject_id', 'hadm_id'] + feature_cols + ['AKI_label', 'center_id', 'split']
df_out = df_final[save_cols].copy()

df_out.to_csv(OUTPUT_CSV, index=False)
file_size = os.path.getsize(OUTPUT_CSV) / 1024 / 1024
print(f'  ✓ Saved: {OUTPUT_CSV}  ({file_size:.2f} MB)  {df_out.shape}')

from google.colab import files
files.download(OUTPUT_CSV)
print('  ✓ Downloaded')


STEP 17: SAVE
  ✓ Saved: aki_anchor_based_24h_lookback.csv  (37.76 MB)  (115404, 94)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ Downloaded


## 18. Validation

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: hours to anchor by AKI status
for label, name, color in [(0, 'Non-AKI', 'steelblue'), (1, 'AKI', 'red')]:
    subset = df_final[df_final['AKI_label'] == label]['hours_to_anchor']
    axes[0].hist(subset.clip(upper=500), bins=50, alpha=0.5,
                 label=f'{name} (n={len(subset):,})', color=color)
axes[0].set_xlabel('Hours from admission to anchor')
axes[0].set_ylabel('N patients')
axes[0].set_title('Time to anchor by AKI status')
axes[0].legend()

# Plot 2: creatinine most_recent distribution by AKI status
if 'creatinine_most_recent' in df_final.columns:
    for label, name, color in [(0, 'Non-AKI', 'steelblue'), (1, 'AKI', 'red')]:
        subset = df_final[df_final['AKI_label'] == label]['creatinine_most_recent'].dropna()
        axes[1].hist(subset.clip(upper=5), bins=50, alpha=0.5,
                     label=f'{name}', color=color, density=True)
    axes[1].set_xlabel('Creatinine most recent (mg/dL)')
    axes[1].set_ylabel('Density')
    axes[1].set_title('SCr distribution by AKI status\n(should differ — sanity check)')
    axes[1].legend()

plt.tight_layout()
plt.savefig('aki_anchor_validation.png', dpi=120)
plt.show()

print('='*70)
print('APPROACH 2 (REVISED) COMPLETE')
print('='*70)
print(f'\n📊 Dataset:')
print(f'   Total patients:         {len(df_final):,}')
print(f'   AKI patients:           {df_final["AKI_label"].sum():,} ({df_final["AKI_label"].mean()*100:.1f}%)')
print(f'   Non-AKI patients:       {(df_final["AKI_label"]==0).sum():,}')
print(f'   Features per patient:   {len(feature_cols)}')
print(f'\n🔬 Methodology:')
print(f'   Anchor (AKI):           First KDIGO-positive SCr')
print(f'   Anchor (non-AKI):       Last SCr during admission')
print(f'   Lookback:               {LOOKBACK_HOURS}h before anchor')
print(f'   Prediction lead time:   ≥{LOOKBACK_HOURS}h before AKI onset')
print(f'   AKI criteria:           KDIGO SCr (≥0.3 rise in 48h OR ≥1.5x baseline)')
print(f'   Train/test split:       patient-level stratified 80/20')
print(f'   Rows per patient:       1 (long-stay patients not over-represented)')
print(f'\n📁 Output: {OUTPUT_CSV}')
print(f'\n⚠️  Next steps:')
print(f'   1. Re-run with LOOKBACK_HOURS=48 for second experiment')
print(f'   2. Update mimic_ftl_simulation_phase2.py — one row per patient,')
print(f'      patient-level Dirichlet sampling (same as Approach 1 structure)')
print(f'   3. fedadapt_train.py requires no changes — flat feature vector per patient')

from google.colab import files
files.download('aki_anchor_validation.png')


APPROACH 2 (REVISED) COMPLETE

📊 Dataset:
   Total patients:         115,404
   AKI patients:           20,316 (17.6%)
   Non-AKI patients:       95,088
   Features per patient:   89

🔬 Methodology:
   Anchor (AKI):           First KDIGO-positive SCr
   Anchor (non-AKI):       Last SCr during admission
   Lookback:               24h before anchor
   Prediction lead time:   ≥24h before AKI onset
   AKI criteria:           KDIGO SCr (≥0.3 rise in 48h OR ≥1.5x baseline)
   Train/test split:       patient-level stratified 80/20
   Rows per patient:       1 (long-stay patients not over-represented)

📁 Output: aki_anchor_based_24h_lookback.csv

⚠️  Next steps:
   1. Re-run with LOOKBACK_HOURS=48 for second experiment
   2. Update mimic_ftl_simulation_phase2.py — one row per patient,
      patient-level Dirichlet sampling (same as Approach 1 structure)
   3. fedadapt_train.py requires no changes — flat feature vector per patient


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>